In [2]:
# Tham số từ bạn
P = 114257609839544410023767157104558613875026989059481832867036707094491730349459
a = P - 1  # = -1 mod P
d = 235
q = 14282201229943051252970894638069826734329386005944649261350544704607507843831
h = 8

# Điểm sinh G
Gx = 23229630557588793144094047753885861212065941738131794352686846971944134455286
Gy = 66049623344638828349163635859251421189934856184538099437413441721098954601166

G = (Gx, Gy)

In [3]:
def inv(x):
    return pow(x, P - 2, P)

def ed_add(P1, P2):
    x1, y1 = P1
    x2, y2 = P2

    denom = inv(1 + d * x1 * x2 * y1 * y2 % P)
    x3 = (x1 * y2 + y1 * x2) * denom % P

    denom = inv(1 - d * x1 * x2 * y1 * y2 % P)
    y3 = (y1 * y2 - a * x1 * x2) * denom % P

    return (x3, y3)

def scalar_mult(k, P):
    R = (0, 1)  # identity
    while k > 0:
        if k & 1:
            R = ed_add(R, P)
        P = ed_add(P, P)
        k >>= 1
    return R

In [4]:
import hashlib

def H(data):
    return hashlib.sha512(data).digest()

def H_int(data):
    return int.from_bytes(H(data), "big") % q

In [5]:
import os

def keygen():
    sk = int.from_bytes(os.urandom(32), "big") % q
    pk = scalar_mult(sk, G)
    return sk, pk

sk, pk = keygen()
print("Private key:", sk)
print("Public key:", pk)

Private key: 4267335842526919892743396467258618402614862617235794699242105004827532577847
Public key: (37237866290849517720471071961421023234812815629615984925419999779745348495154, 39618328257629477147326693926094230392693438673537539776481639382248709058449)


In [6]:
def sign(message, sk, pk):
    sk_bytes = sk.to_bytes(32, "big")
    
    r = H_int(sk_bytes + message)
    R = scalar_mult(r, G)
    
    R_bytes = R[0].to_bytes(32, "big") + R[1].to_bytes(32, "big")
    pk_bytes = pk[0].to_bytes(32, "big") + pk[1].to_bytes(32, "big")
    
    h = H_int(R_bytes + pk_bytes + message)
    
    s = (r + h * sk) % q
    
    return (R, s)

In [7]:
def verify(message, signature, pk):
    R, s = signature
    
    R_bytes = R[0].to_bytes(32, "big") + R[1].to_bytes(32, "big")
    pk_bytes = pk[0].to_bytes(32, "big") + pk[1].to_bytes(32, "big")
    
    h = H_int(R_bytes + pk_bytes + message)
    
    left = scalar_mult(s, G)
    right = ed_add(R, scalar_mult(h, pk))
    
    return left == right

In [8]:
message = b"Hello EdDSA custom curve!"
message2 = b"Hello ae nhe"

signature = sign(message, sk, pk)

print("Signature:")
print("R =", signature[0])
print("s =", signature[1])

valid = verify(message, signature, pk)
print("Valid:", valid)

Signature:
R = (30393898894671431505268568751151270695457382723760977242168525263547569017500, 30829197191732632904682086312622543722261133659621901589859048965769957203705)
s = 12531981809512466752138657693831174982293670168217103222292912386853510851184
Valid: True
